# Preprocesamiento

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Datos

Separamos el archivo de entrenamiento en entrenamiento y validación con `split_train_validation`, y cargamos la prueba con `load_test`.

Todo lo que el preprocesamiento aprende de los datos (categorías conocidas, medias y desviaciones) se ajusta **solo con la parte de entrenamiento**. La validación y la prueba solo se transforman, para comprobar que el preprocesamiento funciona con flujos que no vio.

In [ ]:
from threat_detector.dataset import TARGET, load_test, load_train
from threat_detector.ml.preprocessing import split_train_validation

train_flows = load_train()
train_part, validation_part = split_train_validation(train_flows)
test_flows = load_test()

for name, flows in (("Entrenamiento", train_part), ("Validación", validation_part), ("Prueba", test_flows)):
    print(f"{name}: {len(flows):,} flujos x {flows.shape[1]} columnas; ataques = {flows[TARGET].mean():.1%}")

## 2. Fracción mínima de categorías

El análisis exploratorio conservó las categorías con al menos `MIN_CATEGORY_COUNT` (1.000) flujos en el archivo de entrenamiento completo. Como aquí se ajusta solo con la parte de entrenamiento (el 80 %), el mínimo se expresa como proporción: `MIN_CATEGORY_FRACTION`. Con una cuenta fija de 1.000, `sctp` y `pop3` quedarían por debajo y cambiarían las categorías decididas.

Mostramos la fracción, el corte que da en la parte de entrenamiento y cómo quedan frente a él las tres categorías más cercanas al límite.

In [ ]:
import pandas as pd

from threat_detector.config import MIN_CATEGORY_COUNT, MIN_CATEGORY_FRACTION

cutoff = MIN_CATEGORY_FRACTION * len(train_part)
print(f"MIN_CATEGORY_FRACTION = {MIN_CATEGORY_FRACTION:.5f} ({MIN_CATEGORY_FRACTION:.2%})")
print(f"Corte en la parte de entrenamiento: {cutoff:.0f} flujos (antes: {MIN_CATEGORY_COUNT})")

near_limit = [("proto", "sctp"), ("service", "pop3"), ("service", "ssh")]
pd.DataFrame(
    [
        {
            "variable": col,
            "categoría": category,
            "archivo completo": (train_flows[col] == category).sum(),
            "parte de entrenamiento": (train_part[col] == category).sum(),
            "¿pasa el corte?": (train_part[col] == category).sum() >= cutoff,
        }
        for col, category in near_limit
    ]
)